# PyCharm AI Assistant에서 기존 MCP Server 사용하기

PyCharm의 JetBrains AI Assistant를 Host로 사용해 이미 운영되는 MCP Server를 연결한다. Filesystem으로 로컬 STDIO 연결을, Context7으로 원격 Streamable HTTP 연결을 확인한다.

Server를 직접 구현하는 단계보다 먼저 완성된 Server를 사용하면 `Host → Client → Server` 관계와 설정이 실제로 어떤 역할을 하는지 빠르게 확인할 수 있다.

```mermaid
flowchart LR
    U["사용자"] --> H
    subgraph H["Host · PyCharm JetBrains AI Assistant"]
        CHAT["AI Chat · 모델 · 권한"]
        C1["MCP Client A"]
        C2["MCP Client B"]
        CHAT --> C1
        CHAT --> C2
    end
    C1 -->|"STDIO · Host가 시작"| FS["Filesystem Server"]
    C2 -->|"Streamable HTTP over HTTPS"| CTX["Context7 Server"]
```


## PyCharm AI Assistant의 MCP 설정 열기

JetBrains AI Assistant가 MCP Host가 되어 등록한 Server를 시작하거나 연결하고, AI Chat에서 필요한 Tool을 호출한다.

1. `Settings | Tools | AI Assistant | Model Context Protocol (MCP)`를 연다.
2. `Add`를 눌러 New MCP Server 대화상자를 연다.
3. Server에 맞게 `STDIO` 또는 `HTTP` 연결 방식을 선택한다.
4. JSON configuration에는 최상위 key가 `mcpServers`인 설정을 입력한다.
5. 실습용 Server의 `Server level`은 `Project`로 선택한다.
6. `OK`를 누른 뒤 설정 화면의 `Apply`를 눌러 Server를 시작하거나 연결한다.

Status는 연결 상태를 표시하고, Status 열의 Tool 목록 아이콘은 Server가 공개한 Tool을 보여 준다. 설정을 바꾸거나 연결이 끊어진 뒤에는 해당 Server를 선택하고 `Reconnect`를 누른다.

> 조직에서 AI 기능을 중앙 관리하면 Server가 미리 등록되어 있거나 사용자가 새 Server를 추가하지 못할 수 있다. 메뉴와 Add 버튼이 보이지 않으면 버전보다 조직 정책을 함께 확인한다.


## Node.js와 npx 준비하기

Filesystem Server는 `npx`로 실행하는 Node.js 기반 프로그램이다. [Node.js 공식 다운로드](https://nodejs.org/en/download)에서 **LTS**를 설치한 뒤 PyCharm을 완전히 다시 실행한다.

PyCharm Terminal에서 다음 세 명령의 버전이 모두 출력되는지 확인한다.

```bash
node --version
npm --version
npx --version
```

Python 가상환경과 Node.js는 별도이다. Node.js는 Filesystem과 선택 Inspector에만 필요하다. Context7은 AI Assistant가 원격 HTTP Server에 연결하고, Math·Helpdesk는 Python SDK로 진행한다.


## Filesystem용 실습 폴더 확인하기

PyCharm에서 `09_mcp` 폴더 자체를 project로 연다. Filesystem Server에는 폐기 가능한 `mcp_sandbox` 하나만 허용하고 홈·Downloads·전체 저장소는 허용하지 않는다.

아래 셀의 `working directory`는 STDIO Server 설정의 Working directory에 입력할 값이다. JSON에는 상대 경로 `mcp_sandbox`를 사용하므로 개인 절대 경로를 예시 설정에 저장하지 않는다.

준비된 네 개의 경로 변수 뒤에서 `sandbox_directory`와 `practice_file`이 실제로 존재하는지 검증한다. 이어서 Working directory, 상대 sandbox 경로와 파일명을 출력해 AI Assistant 설정에 사용할 값이 맞는지 확인한다.


In [ ]:
from pathlib import Path

lesson_directory = Path.cwd().resolve()
sandbox_directory = lesson_directory / 'mcp_sandbox'
practice_file = sandbox_directory / 'mcp_practice_note.txt'
sandbox_relative_path = sandbox_directory.relative_to(lesson_directory)


## 공식 출처에서 Server 확인하기

[Smithery](https://smithery.ai/)에서 `filesystem`과 `context7`을 검색해 이름·publisher·연결 방식만 확인한다. 검색 결과는 설치 보증이 아니므로 Filesystem은 [Model Context Protocol 공식 Server](https://github.com/modelcontextprotocol/servers/tree/main/src/filesystem), Context7은 [Upstash 공식 저장소](https://github.com/upstash/context7)와 다시 대조한다.

Server를 찾은 뒤 공식 package와 endpoint를 AI Assistant에 연결한다.


## Filesystem Server를 AI Assistant에 등록하기

Filesystem Server는 로컬 STDIO 방식이다. AI Assistant가 `npx` 명령으로 Server subprocess를 시작하고 마지막 인자로 받은 폴더 안에서만 파일 Tool을 제공한다.

MCP 설정 화면에서 `Add`와 `STDIO`를 선택한 뒤 다음 JSON을 입력한다. `Working directory`에는 앞 셀의 `working directory`를 입력하고, `Server level`은 `Project`를 선택한다.

```json
{
  "mcpServers": {
    "courseFilesystem": {
      "command": "npx",
      "args": [
        "-y",
        "@modelcontextprotocol/server-filesystem@2026.7.10",
        "mcp_sandbox"
      ]
    }
  }
}
```

`command`는 실행할 프로그램이고 `args`는 그 프로그램에 전달할 인자 목록이다. `-y`는 최초 package 실행 확인을 자동 승인하고, package 뒤의 `mcp_sandbox`는 Server가 접근할 수 있는 유일한 실습 폴더가 된다.

`OK`와 `Apply`를 누른 뒤 Status와 Tool 목록을 확인한다. AI Chat에서 다음처럼 요청한다.

> courseFilesystem Tool로 허용된 폴더를 먼저 확인하고, `mcp_practice_note.txt`만 읽어 한 줄로 요약해 줘. 파일은 수정하지 마.

요약 결과보다 먼저 허용 폴더가 `mcp_sandbox` 하나인지 확인한다. 연결 성공과 안전한 접근 범위는 서로 다른 검사항목이다.


## Context7 Server를 AI Assistant에 등록하기

Context7은 외부 운영자가 실행하는 remote Server이다. AI Assistant는 별도 로컬 프로세스를 시작하지 않고 HTTPS로 공개된 Streamable HTTP endpoint에 연결한다.

MCP 설정 화면에서 `Add`와 `HTTP`를 선택한 뒤 다음 JSON을 입력한다.

```json
{
  "mcpServers": {
    "context7": {
      "url": "https://mcp.context7.com/mcp"
    }
  }
}
```

`url`은 AI Assistant가 연결할 Streamable HTTP endpoint이다. `Server level`을 `Project`로 선택하고 `OK`와 `Apply`를 누른 뒤 Status를 확인한다.

AI Chat에서는 공개 라이브러리 문서만 질문한다.

> Context7 Tool을 사용해 LangChain 1.x의 `create_agent` 기본 사용법을 찾고, 핵심 함수와 입력 메시지 형식만 설명해 줘.

질문과 검색 문맥은 외부 서비스로 전송될 수 있다. 내부 코드·고객 정보·API key를 Context7 질문에 포함하지 않는다.


## Filesystem과 Context7 비교하기

두 Server는 같은 `mcpServers` 형식으로 등록하지만 실행 위치와 데이터 경계가 다르다.

| 비교 기준 | Filesystem | Context7 |
|---|---|---|
| 공식 transport | STDIO | Streamable HTTP |
| Server 실행 주체 | AI Assistant가 local subprocess를 시작한다. | 외부 운영자가 remote Server를 실행한다. |
| 연결 대상 | `command`와 `args` | HTTPS `url` |
| 다루는 데이터 | 허용한 로컬 sandbox | 외부 라이브러리 문서 |
| 핵심 위험 | 허용 폴더를 너무 넓게 지정하는 문제 | 내부 정보를 외부 질문으로 보내는 문제 |

Context7의 HTTPS는 별도의 세 번째 transport가 아니라 원격 배포된 Streamable HTTP를 TLS로 보호한 형태이다. 두 경우 모두 AI Assistant가 Host이고, 각 설정 항목에 대응하는 내부 MCP Client가 Server와 통신한다.
